# DATA 255 Lab 2: Same-Resolution Residual Attention Model
**Student Name:** [Your Name]  
**Date:** Spring 2026

## Lab Overview
This notebook implements a same-resolution image restoration model for paired RGB `256x256` images. It covers dataset validation, paired loading, model definition, training, validation PSNR, checkpoint saving, and ONNX export.

### Requirements
1. **Input Size:** `[256, 256, 3]`
2. **Output:** restored RGB image with the same resolution
3. **Constraints:**
   * Use paired LR/HR data with matching basenames.
   * Keep the model export-safe for ONNX.
   * Use validation PSNR to select the best checkpoint.


In [1]:
from contextlib import nullcontext
from pathlib import Path
import json
import math
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

from src.data import (
    PairedImageDataset,
    collect_paired_samples,
    load_rgb_tensor,
    require_canonical_split_dirs,
)

# Check device support
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")


Using device: cpu


## Step 1: Dataset Setup
The dataset should already use the canonical split names:

- `training_data/LR_train`
- `training_data/HR_train`
- `training_data/LR_val`
- `training_data/HR_val`

If any of these folders are missing, the dataset layout is not ready for training.


In [2]:
# Dataset paths
DATA_ROOT = Path("training_data")

# Output path
OUTPUT_DIR = Path("runs/same_resolution_residual_attention_pipeline")

split_dirs = require_canonical_split_dirs(DATA_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_path in split_dirs.items():
    print(f"{split_name}: {split_path}")


LR_train: training_data/LR_train
HR_train: training_data/HR_train
LR_val: training_data/LR_val
HR_val: training_data/HR_val


In [3]:
# Verify train and validation splits
def verify_split(lr_dir: Path, hr_dir: Path, split_name: str) -> None:
    pairs = collect_paired_samples(lr_dir, hr_dir)
    for sample in pairs:
        load_rgb_tensor(sample.lr_path)
        load_rgb_tensor(sample.hr_path)
    print(f"[{split_name}] pairs={len(pairs)} | all images load as RGB 256x256")

verify_split(split_dirs["LR_train"], split_dirs["HR_train"], "train")
verify_split(split_dirs["LR_val"], split_dirs["HR_val"], "val")


[train] pairs=810 | all images load as RGB 256x256
[val] pairs=781 | all images load as RGB 256x256


## Step 2: Data Loading
The paired dataset loads RGB images, converts them to tensors in `[0, 1]`, and keeps LR/HR transforms synchronized during training.


In [4]:
# Hyperparameters
BATCH_SIZE = 8
NUM_WORKERS = max(0, min(4, (os.cpu_count() or 1) - 1))
SEED = 255

def build_loader_kwargs(num_workers: int) -> dict:
    loader_kwargs = {}
    if device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2
    return loader_kwargs

def build_loaders(num_workers: int) -> tuple[DataLoader, DataLoader, dict]:
    loader_kwargs = build_loader_kwargs(num_workers)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=num_workers,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=num_workers,
        **loader_kwargs,
    )
    return train_loader, val_loader, loader_kwargs

# Create datasets
train_dataset = PairedImageDataset(
    split_dirs["LR_train"],
    split_dirs["HR_train"],
    train=True,
    seed=SEED,
)
val_dataset = PairedImageDataset(
    split_dirs["LR_val"],
    split_dirs["HR_val"],
    train=False,
    seed=SEED,
)

# Create data loaders
train_loader, val_loader, loader_kwargs = build_loaders(NUM_WORKERS)

# Inspect one batch
try:
    lr_batch, hr_batch, names = next(iter(train_loader))
except RuntimeError as exc:
    if NUM_WORKERS == 0:
        raise
    print(f"multi-worker data loading failed ({exc}); retrying with num_workers=0")
    NUM_WORKERS = 0
    train_loader, val_loader, loader_kwargs = build_loaders(NUM_WORKERS)
    lr_batch, hr_batch, names = next(iter(train_loader))

print("train dataset size:", len(train_dataset))
print("val dataset size:", len(val_dataset))
print("num workers:", NUM_WORKERS)
print("loader kwargs:", loader_kwargs)
print("lr batch shape:", tuple(lr_batch.shape), lr_batch.dtype, float(lr_batch.min()), float(lr_batch.max()))
print("hr batch shape:", tuple(hr_batch.shape), hr_batch.dtype, float(hr_batch.min()), float(hr_batch.max()))
print("sample basenames:", list(names[:3]))


Python(14557) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(14558) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(14559) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(14560) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(14561) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


train dataset size: 810
val dataset size: 781
num workers: 4
loader kwargs: {'persistent_workers': True, 'prefetch_factor': 2}
lr batch shape: (8, 3, 256, 256) torch.float32 0.0 1.0
hr batch shape: (8, 3, 256, 256) torch.float32 0.0 1.0
sample basenames: ['000392', '000469', '000237']


## Step 3: Model Definition
The model keeps the input and output at the same resolution. It uses a convolution stem, residual groups with attention, and a final residual image add.


In [5]:
# Forbidden layers for this model
FORBIDDEN_TYPES = (
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.BatchNorm3d,
    nn.ReLU,
    nn.Sigmoid,
    nn.Softmax,
    nn.Upsample,
)

def conv3x3(in_channels: int, out_channels: int) -> nn.Conv2d:
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

# Residual block
class ResidualBlock(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.expand = nn.Conv2d(48, 96, kernel_size=1)
        self.depthwise = nn.Conv2d(96, 96, kernel_size=kernel_size, padding=padding, groups=96)
        self.act = nn.PReLU(num_parameters=96)
        self.project = nn.Conv2d(96, 48, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.expand(x)
        x = self.depthwise(x)
        x = self.act(x)
        x = self.project(x)
        return residual + x

# Stage attention
class StageAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.reduce = nn.Conv2d(48, 12, kernel_size=1)
        self.act = nn.PReLU(num_parameters=12)
        self.expand = nn.Conv2d(12, 48, kernel_size=1)
        self.gate = nn.Hardsigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = self.reduce(scale)
        scale = self.act(scale)
        scale = self.expand(scale)
        scale = self.gate(scale)
        return x * scale

# Residual group
class ResidualGroup(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(
            ResidualBlock(kernel_size=3),
            ResidualBlock(kernel_size=5),
            ResidualBlock(kernel_size=7),
        )
        self.attention = StageAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.blocks(x)
        x = self.attention(x)
        return residual + x

# Main model
class MultiScaleResidualAttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            conv3x3(3, 48),
            nn.PReLU(num_parameters=48),
        )
        self.body = nn.Sequential(
            ResidualGroup(),
            ResidualGroup(),
            ResidualGroup(),
        )
        self.tail = nn.Sequential(
            conv3x3(48, 24),
            nn.PReLU(num_parameters=24),
            conv3x3(24, 3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        image_residual = x
        stem_features = self.stem(x)
        body_features = self.body(stem_features)
        fused = body_features + stem_features
        delta = self.tail(fused)
        return image_residual + delta

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def assert_no_forbidden_modules(model: nn.Module) -> None:
    for module in model.modules():
        if isinstance(module, FORBIDDEN_TYPES):
            raise TypeError(f"Forbidden module found: {module.__class__.__name__}")

# Initialize the model
model = MultiScaleResidualAttentionNet().to(device)
assert_no_forbidden_modules(model)

print("parameter count:", count_parameters(model))


parameter count: 126003


## Step 4: Loss and Training
Training uses L1 loss. Validation uses PSNR on clamped `[0, 1]` predictions, and the best checkpoint is selected by validation PSNR.


In [6]:
# Training settings
TRAIN_CFG = {
    "epochs": 80,
    "batch_size": 8,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "min_lr_ratio": 0.05,
    "num_workers": NUM_WORKERS,
    "seed": SEED,
    "use_amp": device.type == "cuda",
}
print(TRAIN_CFG)

# Set random seed
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# PSNR metric
def compute_psnr(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(eps)
    return -10.0 * torch.log10(mse)

# Learning rate schedule
def lr_for_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int, min_lr_ratio: float) -> float:
    if total_epochs <= 0:
        raise ValueError("total_epochs must be positive")
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    decay_epochs = max(1, total_epochs - warmup_epochs)
    progress = float(epoch - warmup_epochs) / float(decay_epochs - 1 if decay_epochs > 1 else 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_lr_ratio
    return min_lr + (base_lr - min_lr) * cosine

# Update optimizer learning rate
def set_optimizer_lr(optimizer: torch.optim.Optimizer, lr: float) -> None:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def autocast_context(enabled: bool):
    return torch.cuda.amp.autocast(enabled=True) if enabled else nullcontext()

# Train for one epoch
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: torch.cuda.amp.GradScaler | None = None,
    use_amp: bool = False,
) -> float:
    model.train()
    total_loss = 0.0
    total_items = 0
    amp_enabled = bool(use_amp and device.type == "cuda" and scaler is not None)

    for lr_img, hr_img, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context(amp_enabled):
            pred = model(lr_img)
            loss = F.l1_loss(pred, hr_img)

        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = lr_img.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    return total_loss / max(1, total_items)

# Validate for one epoch
def validate(model: nn.Module, loader: DataLoader, device: torch.device, use_amp: bool = False) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_items = 0
    amp_enabled = bool(use_amp and device.type == "cuda")

    with torch.inference_mode():
        for lr_img, hr_img, _ in loader:
            lr_img = lr_img.to(device, non_blocking=True)
            hr_img = hr_img.to(device, non_blocking=True)

            with autocast_context(amp_enabled):
                pred = model(lr_img)
                loss = F.l1_loss(pred, hr_img)
            psnr = compute_psnr(pred.float(), hr_img.float())

            if not torch.isfinite(psnr).all():
                raise FloatingPointError("Validation PSNR produced non-finite values")

            batch_size = lr_img.size(0)
            total_loss += loss.item() * batch_size
            total_psnr += psnr.sum().item()
            total_items += batch_size

    return {
        "val_loss": total_loss / max(1, total_items),
        "val_psnr": total_psnr / max(1, total_items),
    }

# Save checkpoint
def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, metrics: dict) -> None:
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "train_cfg": TRAIN_CFG,
    }
    torch.save(payload, path)

def summarize_history(history: list[dict], output_dir: Path) -> dict[str, float | int | str]:
    if not history:
        raise ValueError("history must contain at least one epoch")

    output_dir = Path(output_dir)
    first_row = history[0]
    final_row = history[-1]
    best_row = max(history, key=lambda row: row["val_psnr"])
    summary = {
        "best_epoch": best_row["epoch"],
        "best_val_psnr": best_row["val_psnr"],
        "final_val_psnr": final_row["val_psnr"],
        "final_val_loss": final_row["val_loss"],
        "val_psnr_gain_to_best": best_row["val_psnr"] - first_row["val_psnr"],
        "val_psnr_gain_to_final": final_row["val_psnr"] - first_row["val_psnr"],
        "avg_epoch_seconds": sum(row["seconds"] for row in history) / len(history),
        "best_checkpoint": str(output_dir / "best.pt"),
        "last_checkpoint": str(output_dir / "last.pt"),
        "metrics_path": str(output_dir / "metrics.jsonl"),
    }
    print("training summary:", summary)
    return summary

# Training loop
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    output_dir: Path,
    epochs: int = TRAIN_CFG["epochs"],
    lr: float = TRAIN_CFG["lr"],
    weight_decay: float = TRAIN_CFG["weight_decay"],
    warmup_epochs: int = TRAIN_CFG["warmup_epochs"],
    min_lr_ratio: float = TRAIN_CFG["min_lr_ratio"],
    seed: int = TRAIN_CFG["seed"],
    use_amp: bool = TRAIN_CFG["use_amp"],
):
    set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    metrics_path.write_text("")

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    amp_enabled = bool(use_amp and device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled) if amp_enabled else None
    best_psnr = float("-inf")
    history = []

    for epoch in range(epochs):
        epoch_lr = lr_for_epoch(epoch, epochs, lr, warmup_epochs, min_lr_ratio)
        set_optimizer_lr(optimizer, epoch_lr)
        start_time = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler=scaler, use_amp=amp_enabled)
        val_metrics = validate(model, val_loader, device, use_amp=amp_enabled)
        elapsed = time.time() - start_time

        row = {
            "epoch": epoch + 1,
            "lr": epoch_lr,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "val_psnr": val_metrics["val_psnr"],
            "seconds": elapsed,
        }
        history.append(row)
        with metrics_path.open("a") as handle:
            handle.write(json.dumps(row) + "\n")

        save_checkpoint(output_dir / "last.pt", model, optimizer, epoch + 1, row)
        if row["val_psnr"] > best_psnr:
            best_psnr = row["val_psnr"]
            save_checkpoint(output_dir / "best.pt", model, optimizer, epoch + 1, row)

        print(
            f"epoch={epoch + 1:03d}/{epochs:03d} | "
            f"lr={epoch_lr:.2e} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={row['val_loss']:.6f} | "
            f"val_psnr={row['val_psnr']:.3f} dB | "
            f"seconds={elapsed:.1f}s"
        )

    return history


{'epochs': 80, 'batch_size': 8, 'lr': 0.0002, 'weight_decay': 0.0001, 'warmup_epochs': 5, 'min_lr_ratio': 0.05, 'num_workers': 4, 'seed': 255, 'use_amp': False}


## Step 5: Full Training and Export
Execute the full training cell below to run the complete train/validation pipeline. Each epoch prints training loss, validation loss, validation PSNR, and elapsed seconds. When training finishes, the notebook prints a compact improvement summary before exporting and verifying the best checkpoint.


In [7]:
# Load checkpoint weights
def load_checkpoint_state(model: nn.Module, checkpoint_path: Path, map_location: str | torch.device = "cpu"):
    checkpoint = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    return checkpoint

# Export model to ONNX
def export_checkpoint_to_onnx(checkpoint_path, onnx_out, sample_image=None, verify=False):
    checkpoint_path = Path(checkpoint_path)
    onnx_out = Path(onnx_out)
    onnx_out.parent.mkdir(parents=True, exist_ok=True)

    model = MultiScaleResidualAttentionNet().to(device)
    load_checkpoint_state(model, checkpoint_path, map_location=device)
    model.eval()

    dummy_input = torch.randn(1, 3, 256, 256, device=device)
    export_kwargs = dict(
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
    )

    try:
        torch.onnx.export(model, dummy_input, onnx_out, dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(model, dummy_input, onnx_out, **export_kwargs)

    print(f"Exported ONNX model to {onnx_out}")

    if onnx is not None:
        onnx_model = onnx.load(str(onnx_out))
        onnx.checker.check_model(onnx_model)
        print("ONNX checker passed")
    else:
        print("onnx package not installed; skipped checker")

    if verify:
        if ort is None:
            raise ImportError("onnxruntime is required for parity verification")
        if sample_image is None:
            raise ValueError("sample_image is required when verify=True")

        sample_tensor = load_rgb_tensor(sample_image).unsqueeze(0).to(device)
        with torch.inference_mode():
            torch_output = model(sample_tensor).cpu().numpy()

        session = ort.InferenceSession(str(onnx_out), providers=["CPUExecutionProvider"])
        ort_output = session.run(["output"], {"input": sample_tensor.cpu().numpy()})[0]
        diff = abs(torch_output - ort_output)

        stats = {
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
        }
        print(stats)
        return stats

    return None


In [ ]:
# Full training run
RUN_FULL_PIPELINE = True

history = None
training_summary = None
if RUN_FULL_PIPELINE:
    history = fit(model, train_loader, val_loader, OUTPUT_DIR)
    training_summary = summarize_history(history, OUTPUT_DIR)
    print("training complete")


In [ ]:
# Export the best checkpoint and verify ONNX parity on a deterministic validation sample
sample_path = sorted(split_dirs["LR_val"].glob("*.png"))[0]
parity_stats = export_checkpoint_to_onnx(
    OUTPUT_DIR / "best.pt",
    OUTPUT_DIR / "best.onnx",
    sample_image=sample_path,
    verify=True,
)
print("export sample:", sample_path.name)
print("parity stats:", parity_stats)


## Run All Behavior
A full top-to-bottom run now executes the notebook in this order:

1. Verify canonical paired dataset folders and image integrity.
2. Build the full train/validation loaders and inspect one batch.
3. Train on the full dataset with `fit(...)`, printing per-epoch training and validation metrics while writing `metrics.jsonl`, `last.pt`, and `best.pt` under `runs/same_resolution_residual_attention_pipeline`.
4. Print a compact training summary showing the best and final validation results plus PSNR improvement and average epoch time.
5. Export `best.pt` to `best.onnx` and verify PyTorch/ONNX Runtime parity on the first validation sample.
